# CodeAlpha Task 4 — Amazon Review Sentiment Analysis

## Objective
Use NLP to classify Amazon cell-phone reviews.

The UCI source dataset contains 500 positive and 500 negative reviews and intentionally excludes neutral examples. For new text, this project returns **Neutral** when the predicted probability is between 0.45 and 0.55.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv(Path("data/amazon_reviews.csv"))
df.head()

In [ ]:
df["sentiment"].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"],
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True
    )),
    ("model", LogisticRegression(
        max_iter=2000,
        C=2.0,
        random_state=42
    ))
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print(classification_report(
    y_test,
    predictions,
    target_names=["Negative", "Positive"]
))

In [ ]:
def predict_sentiment(text):
    probability = float(model.predict_proba([text])[0, 1])
    if probability >= 0.55:
        label = "Positive"
    elif probability <= 0.45:
        label = "Negative"
    else:
        label = "Neutral"

    return {
        "text": text,
        "sentiment": label,
        "positive_probability": round(probability, 4),
    }

for sample in [
    "The product is excellent and works perfectly.",
    "It is okay, neither great nor terrible.",
    "The battery is poor and the phone stopped working."
]:
    print(predict_sentiment(sample))

## Completed result
- Reviews: **1,000**
- Positive: **500**
- Negative: **500**
- Test accuracy: **82.00%**
- Weighted F1-score: **0.8199**

![Sentiment distribution](outputs/sentiment_distribution.png)

![Confusion matrix](outputs/confusion_matrix.png)

![Review-length distribution](outputs/review_length_distribution.png)